# Agrégation Adhérents - aCréation de la Base Adhérents

Ce notebook consolide la base contacts nettoyée (`clean_base_membres.xlsx`) pour construire la table analytique finale **`agregation_adherents`** à la granularité stricte d'**une ligne par adhérent** (`GROUPE - ID`).


### 1. Entrées & Sorties

| Type | Fichier / Répertoire | Description |
| :--- | :--- | :--- |
| **Entrée** | `../data/clean_base_membres.xlsx` | Base contacts nettoyée (6 685 individus rattachés à leurs organisations). |
| **Sortie (Données)** | `../data/base_adherents.xlsx` | Table finale prête pour la modélisation (1 ligne / organisation adhérente). |
| **Sortie (Audit)** | `../outputs/rapports/rapport_base_adherents_*.html` | Rapports d'intégrité et de distribution par blocs (`skrub.TableReport`). |


### 2. Logique d'Agrégation par Dimension

| Dimension | Règle d'agrégation appliquée | Variables résultantes |
| :--- | :--- | :--- |
| **Identité & Cotisations** | Mode majoritaire (`first_mode`) | Années adhésion/démission, montants, statut adhésion, secteur. |
| **Profil & Influence** | Présence d'au moins un contact (`_any_true`) | `has_C_level`, `has_M_level`, `has_F_level`, `has_VIP`, `has_active_speaker`. |
| **Départements** | Consensus à 3 états (`agg_tri_state`) | Couverture des 8 directions métier (`department_*`). |
| **Effectifs & Récence** | Décomptes d'individus et écarts temporels | `nb_contacts`, `nb_current_contacts`, ancienneté (min/max). |
| **Engagement Emailing** | Moyennes et minima par organisation | Taux d'ouverture/clic moyens, annualisation, récence des ouvertures. |
| **Événements & Réunions** | Moyennes d'assiduité et d'anticipation | `avg_presence_rate`, volume annuel d'inscriptions, réactivité aux invitations. |

---

### 3. Pipeline d'Exécution  

1. Agrégation du profil adhérent (Mode & arbitrage des conflits)
2. Consolidation de l'équipe (Flags C-Level, VIP, Départements)
3. Volumétrie et ancienneté des contacts rattachés
4. Métriques moyennes d'engagement (Emailing & Meetings)
5. Récence des dernières inter

# Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd

from skrub import TableReport

from src.utils import convert_to_date_day_first
from src.agregation_adherents import (
    GROUP_COL,
    first_mode,
    detect_conflicts,
    aggregate_columns_by_mode,
    resolve_value_conflicts_interactive,
    add_any_true_flags,
    aggregate_department_flags,
    count_members,
    count_members_by_status,
    active_contact_date_range,
    categorical_rates_by_group,
    recency_by_group,
    average_anticipation_days_by_group,
    acceptance_counts_and_rates_by_group,
    numeric_agg_by_group,
    contact_date_range,
    add_numeric_aggregations,
    aggregate_membership_status,
    unsubscribe_rate_by_group,
    _any_true,
    categorical_rate_recipient_status,
    build_acceptance_mask,
    numeric_agg_by_group
)


# Chargement des données

In [ ]:
df = pd.read_excel("../data/base_membres_propre.xlsx")
print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

# 1. Agrégation du profil adhérent (Mode & arbitrage des conflits)

Agrégation par **mode** (valeur la plus fréquente) pour chaque variable —
crée `df_marque`, la table de sortie (une ligne par adhérent).

In [ ]:
info_marque_cols = [
    "GROUPE - Année adhésion *",
    "GROUPE - Année démission *",
    "GROUPE - Nombre d'adhésion",
    "duree_derniere_adhesion",
    "Montant adhésion",
    "Montant démission",
    "a_deja_demissionne",
    "GROUPE - Nom",
    'Organisation - Sector of Activities',
    "Organisation - Marketing Budget 2019",
    "target",
]

df_marque = aggregate_columns_by_mode(df, GROUP_COL, info_marque_cols)
df_marque.shape


In [ ]:
membership_agg = aggregate_membership_status(df, GROUP_COL, 'Organisation - Membership status')
df_marque = df_marque.merge(membership_agg, on=GROUP_COL, how='left', suffixes=('_old', ''))

**Suppression des organisations n'étant plus membres en 2019**

In [ ]:
mask_remove = (
    (df_marque["Organisation - Membership status"] == "Former Member")
    & (pd.to_numeric(df_marque["GROUPE - Année démission *"], errors="coerce") <= 2019)
)

df_marque = df_marque.loc[~mask_remove].copy()
df_marque.shape

In [ ]:
def aggregate_dummy_columns_by_group(df, GROUP_COL, prefix='secteur_'):
    cols = [c for c in df.columns if c.startswith(prefix)]
    if not cols:
        raise ValueError(f"Aucune colonne ne commence par {prefix!r} dans le DataFrame")

    return (
        df[[GROUP_COL] + cols]
        .groupby(GROUP_COL, dropna=False)
        .agg(lambda s: int((s == 1).any()))
        .reset_index()
    )

secteur_dummies = aggregate_dummy_columns_by_group(df, GROUP_COL, prefix='secteur_')
df_marque = df_marque.merge(secteur_dummies, on=GROUP_COL, how='left')

### [Optionnel / exploration] Résolution interactive des conflits sur 'Company name'

Cellule interactive (`input()`), utile uniquement en exploration ponctuelle
pour arbitrer manuellement les valeurs en conflit. Désactivée par défaut
(`RESOLVE_COMPANY_NAME_CONFLICTS = False`) — ne pas exécuter en mode batch.

In [ ]:
RESOLVE_COMPANY_NAME_CONFLICTS = False  # passer à True pour lancer l'arbitrage interactif

if RESOLVE_COMPANY_NAME_CONFLICTS and 'Company name' in df.columns:
    df_company = df[[GROUP_COL, 'Company name']].copy()
    modified_ids = resolve_value_conflicts_interactive(df_company, df_marque, GROUP_COL, 'Company name')
    print(f"{len(modified_ids)} relations modifiées.")


# 2. Consolidation de l'équipe (Flags C-Level, VIP, Départements)

## Profils de contacts (F/M/C level, VIP, Speaker)

In [ ]:
level_flags = {
    'bool_Is F LEVEL': 'has_F_level',
    'bool_Is M LEVEL': 'has_M_level',
    'bool_Is C LEVEL': 'has_C_level',
    'VIP': 'has_VIP',
    'Meeting&Events speaker': 'has_active_speaker'
}

df_marque = add_any_true_flags(df, df_marque, GROUP_COL, level_flags)
df_marque[[GROUP_COL] + list(level_flags.values())].head(20)


## Colonnes département (`department_*`)

In [ ]:
df_dept, ambiguous_departments = aggregate_department_flags(df, GROUP_COL)
df_marque = df_marque.merge(df_dept, on=GROUP_COL, how='left')

if ambiguous_departments:
    print("Groupes ambigus (False + NaN sans aucun True) par colonne :")
    for col, ids in ambiguous_departments.items():
        print(f" - {col}: {len(ids)} groupes, ex. {ids[:10]}")
else:
    print("Aucun groupe ambigu détecté.")

---

## Nombre de membres actifs (Current Member)

In [ ]:
current_counts = count_members(df, GROUP_COL)
df_marque = df_marque.merge(current_counts, on=GROUP_COL, how='left')
df_marque['nb_contacts'] = df_marque['nb_contacts'].fillna(0).astype(int)

df_marque[[GROUP_COL, 'nb_contacts']].head(20)

Nombre de contacts ayant reçus au moins un mail entre 2019 et 2025.

In [ ]:
mail_counts = count_members(
    df[
        df["rows_count_by_relation"].notna()
        & (df["rows_count_by_relation"].astype(str).str.strip() != "")
    ],
    group_col=GROUP_COL,
    new_name="nb_contacts_ayant_recu_mail"
)
df_marque = df_marque.merge(mail_counts, on=GROUP_COL, how="left")
df_marque["nb_contacts_ayant_recu_mail"] = (
    df_marque["nb_contacts_ayant_recu_mail"]
    .fillna(0)
    .astype(int)
)

In [ ]:
status_col = 'Membership status (description)'
org_status_col = 'Organisation - Membership status'

current_counts = count_members_by_status(df, GROUP_COL, status_col, org_status_col, new_name="nb_current_contacts")
df_marque = df_marque.merge(current_counts, on=GROUP_COL, how='left')
df_marque['nb_current_contacts'] = df_marque['nb_current_contacts'].fillna(0).astype(int)

df_marque[[GROUP_COL, 'nb_current_contacts']].head(20)


### Ancienneté des contacts actifs (le plus ancien / le plus récent)

pas forcément actif

In [ ]:
date_range = contact_date_range(df, GROUP_COL, date_col='Creation date')
df_marque = df_marque.merge(date_range, on=GROUP_COL, how='left')

df_marque[[GROUP_COL, 'creation_date_oldest_contact', 'creation_date_youngest_contact']].head(20)

actif

In [ ]:
date_range = active_contact_date_range(df, GROUP_COL, status_col, org_status_col, date_col='Creation date')
df_marque = df_marque.merge(date_range, on=GROUP_COL, how='left')

df_marque[[GROUP_COL, 'creation_date_oldest_current_member', 'creation_date_youngest_current_member']].head(20)


### Taux par Recipient Status

In [ ]:
recipient_status_rates = categorical_rates_by_group(df, GROUP_COL, 'Recipient Status', prefix='recipient_status')
df_marque = df_marque.merge(recipient_status_rates, on=GROUP_COL, how='left')

recipient_status_rates.head()


## 4. Activité & engagement

### Récence de dernière connexion

In [ ]:
convert_to_date_day_first(df, 'User last online')

recency_online = recency_by_group(df, GROUP_COL, 'User last online', new_name='days_since_last_online')
df_marque = df_marque.merge(recency_online, on=GROUP_COL, how='left')

df_marque[[GROUP_COL, 'days_since_last_online']].sample(5)


### Autorisations de communication (niveau organisation)

Ces variables `Organisation - Communication - *` sont censées être
constantes par organisation : on vérifie l'absence de conflit avant
d'agréger par mode.

In [ ]:
comm_cols = [c for c in df.columns if c.startswith('Organisation - Communication')]
print(f"{len(comm_cols)} colonnes 'Organisation - Communication - *' trouvées")

_, comm_conflicts = detect_conflicts(df, GROUP_COL, comm_cols)
flagged = {c: n for c, n in comm_conflicts.items() if n}
print("Colonnes avec plusieurs valeurs par organisation :", flagged if flagged else "aucune")

comm_aggr = df.groupby(GROUP_COL, dropna=False)[comm_cols].agg(first_mode).reset_index()
df_marque = df_marque.merge(comm_aggr, on=GROUP_COL, how='left')


### Taux d'acceptation des communications (niveau contact)

> ⚠️ **Changement** : calculé mais jamais fusionné dans `df_marque` dans la
> version d'origine. Fusion ajoutée ici (colonnes préfixées `comm_accept_`).

In [ ]:
comm_contact_cols = [c for c in df.columns if c.startswith('Communication - ')]

acceptance = acceptance_counts_and_rates_by_group(df, GROUP_COL, comm_contact_cols)
df_marque = df_marque.merge(acceptance, on=GROUP_COL, how='left')

acceptance.head()


### Indicateurs moyens d'engagement par contact

Tous calculés sur le même schéma (moyenne par groupe, conversion numérique
robuste) — regroupés ici en une seule liste de configuration plutôt qu'une
cellule dupliquée par variable.

In [ ]:
mean_engagement_specs = [
    ('clicks_count_by_relation', 'avg_clicks_per_contact'),
    ('rows_count_by_relation', 'avg_mails_received_per_contact'),
    ('nb_mails_open_by_relation', 'avg_mails_open_per_contact'),
    ('nb_mails_clicked_by_relation', 'avg_mails_clicked_per_contact'),
    ('taux_ouverture', 'avg_taux_ouverture'),
    ('taux_click','avg_taux_click'),
    ('taux_click_sur_ouverture','avg_taux_click_sur_overture'),
    ('days_since_last_open_by_relation', 'avg_days_since_last_open'),
    ('days_since_last_participation', 'avg_days_since_last_participation'),
    ('days_since_last_registration', 'avg_days_since_last_registration'),
    ('average_days_open_by_relation', 'avg_days_open_per_contact'),
    ('average_open_delay_hours', 'avg_open_delay_hours_by_contact'),
    ('present_count_by_relation', 'avg_present_count_per_contact'),
    ('registered_count_by_relation', 'avg_registered_count_per_contact'),
    ('presence_rate_by_relation', 'avg_presence_rate_by_relation'),
    ('days_since_last_click', 'avg_days_since_last_click'),
    ('spontaneous_participation_rate_by_relation', 'avg_spontaneous_participation_rate_by_relation'),
    ('invitation_reactivity_by_relation', 'avg_invitation_reactivity_by_relation'),
    ('cancelled_count_by_relation', 'avg_cancelled_count_by_relation'),
    ('opted_out_count_by_relation', 'avg_opted_out_count_by_relation'),
    ('reserve_list_count_by_relation', 'avg_reserve_list_count_by_relation'),
    ('no_reaction_count_by_relation', 'avg_no_reaction_count_by_relation')
]

df_marque = add_numeric_aggregations(df, df_marque, GROUP_COL, mean_engagement_specs, how='mean')
df_marque[[s[1] for s in mean_engagement_specs]].describe()


In [ ]:
df_marque = add_numeric_aggregations(df, df_marque, GROUP_COL, [('registered_count_by_relation','registered_count_by_organisation')], how='sum')

In [ ]:
def calculate_annual_rate(row, value_col):
    """
    Calcule le nombre moyen de mails reçus par an.
    Période de référence : 2019 à 2025 (7 ans).
    Ajuste selon l'année d'adhésion et de démission.
    """  
    value = row[value_col]
  
    if pd.isna(value):
        return pd.NA
    
    adhesion_year = pd.to_numeric(row['GROUPE - Année adhésion *'], errors='coerce')
    resignation_year = pd.to_numeric(row['GROUPE - Année démission *'], errors='coerce')
    
    # Calcul de la durée en années
    if pd.notna(adhesion_year) and adhesion_year > 2019 and pd.notna(resignation_year):
        duration_years = resignation_year - adhesion_year
    elif pd.notna(resignation_year):
        duration_years = resignation_year - 2019
    elif pd.notna(adhesion_year) and adhesion_year > 2019:
        duration_years = 2025 - adhesion_year + 1
    else:
        duration_years = 7  # Période complète 2019-2025
    
    duration_years = max(duration_years, 1)  # Au minimum 1 an
    
    return round(value / duration_years, 2)

df_marque['nb_moyen_emails_recus_par_contact_par_an'] = df_marque.apply(
    lambda row: calculate_annual_rate(
        row,
        'avg_mails_received_per_contact'
    ),
    axis=1
)

df_marque['nb_moyen_inscriptions_par_contact_par_an'] = df_marque.apply(
    lambda row: calculate_annual_rate(
        row,
        'avg_registered_count_per_contact'
    ),
    axis=1
)

df_marque['nb_inscriptions_par_organisation_par_an'] = df_marque.apply(
    lambda row: calculate_annual_rate(
        row,
        'registered_count_by_organisation'
    ),
    axis=1
)

df_marque[[GROUP_COL, 'avg_mails_received_per_contact', 'nb_moyen_emails_recus_par_contact_par_an']].head(20)

### Récence des dernières interactions (clic, ouverture, participation, inscription)

Même principe : un minimum par groupe (le plus petit nombre de jours =
l'évènement le plus récent), regroupé en une seule liste de configuration.

> ⚠️ **Changement** : ces trois indicateurs (`days_since_most_recent_open`,
> `_participation`, `_registration`) étaient calculés mais jamais fusionnés
> dans `df_marque` dans la version d'origine. Fusion ajoutée ici.

In [ ]:
recency_specs = [
    ('days_since_last_click', 'days_since_most_recent_click'),
    ('days_since_last_open_by_relation', 'days_since_most_recent_open'),
    ('days_since_last_participation', 'days_since_most_recent_participation'),
    ('days_since_last_registration', 'days_since_most_recent_registration'),
]

df_marque = add_numeric_aggregations(df, df_marque, GROUP_COL, recency_specs, how='min')
df_marque[[s[1] for s in recency_specs]].describe()


### Désabonnement

In [ ]:
unsub = unsubscribe_rate_by_group(df, GROUP_COL, 'has_unsubscribed_by_relation')

df_marque = df_marque.merge(
    unsub[[GROUP_COL, 'nb_unsubscribed_by_relation', 'unsub_rate_pct']],
    on=GROUP_COL, how='left'
).fillna({'nb_unsubscribed_by_relation': 0, 'unsub_rate_pct': 0.0})

unsub.sort_values('unsub_rate_pct', ascending=False).head(10)


### Nombre moyen de jours d'anticipation

In [ ]:
anticipation = average_anticipation_days_by_group(df, GROUP_COL)
df_marque = df_marque.merge(anticipation, on=GROUP_COL, how='left')

# 7. Exports & Audit qualité

In [ ]:
OUTPUT_EXCEL = "../data/base_adherents.xlsx"
OUTPUT_HTML  = "../outputs/rapports/rapport_base_adherents.html"

chunk_size = 30
reports = []

for part, start in enumerate(range(0, len(df_marque.columns), chunk_size), start=1):
    cols = df_marque.columns[start : start + chunk_size]
    chunk_report = TableReport(df_marque[cols])
    output_part_html = OUTPUT_HTML.replace(".html", f"_part{part:02d}.html")
    chunk_report.write_html(output_part_html)
    print(f"Rapport HTML écrit : {output_part_html}")
    reports.append(chunk_report)

report = reports[0]

# Export Excel
df_marque.to_excel(OUTPUT_EXCEL, index=False)
print(f"Export Excel écrit : {OUTPUT_EXCEL}")

report